# Lesson 9.4 - Domain AI: Applied AI in Key Industries (case study notebook)

## Objectives

- Compare AI system design across healthcare, finance, and smart-city contexts.
- Build lightweight toy pipelines for domain framing.
- Understand domain-specific risk and evaluation differences.


In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

np.random.seed(42)


## Domain Case Study A: Finance (Fraud Screening Toy Model)

### Problem

Classify potentially fraudulent transactions for triage.


In [2]:
n = 1500
df = pd.DataFrame({
    "amount": np.random.exponential(120, n),
    "velocity_1h": np.random.poisson(2, n),
    "is_new_device": np.random.binomial(1, 0.25, n),
    "geo_mismatch": np.random.binomial(1, 0.15, n),
})

logit = (
    0.01 * df["amount"]
    + 0.6 * df["velocity_1h"]
    + 1.1 * df["is_new_device"]
    + 1.5 * df["geo_mismatch"]
    - 4.0
)
prob = 1 / (1 + np.exp(-logit))
df["fraud"] = (np.random.rand(n) < prob).astype(int)

X = df.drop(columns=["fraud"])
y = df["fraud"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=200)
clf.fit(X_train, y_train)

pred_prob = clf.predict_proba(X_test)[:, 1]
pred = (pred_prob > 0.5).astype(int)

print('AUC:', round(roc_auc_score(y_test, pred_prob), 3))
print(classification_report(y_test, pred, digits=3))


AUC: 0.84
              precision    recall  f1-score   support

           0      0.827     0.915     0.869       271
           1      0.693     0.500     0.581       104

    accuracy                          0.800       375
   macro avg      0.760     0.708     0.725       375
weighted avg      0.790     0.800     0.789       375



## Domain Case Study B: Healthcare (Risk Scoring Skeleton)

### Problem

Estimate short-term deterioration risk from vitals/lab proxies.


In [3]:
m = 1200
health = pd.DataFrame({
    "heart_rate": np.random.normal(88, 14, m),
    "resp_rate": np.random.normal(20, 4, m),
    "spo2": np.random.normal(95, 2.5, m),
    "crp": np.random.gamma(shape=2, scale=3, size=m),
})

risk_score = (
    0.03 * (health["heart_rate"] - 80)
    + 0.08 * (health["resp_rate"] - 18)
    + 0.20 * np.maximum(0, 94 - health["spo2"])
    + 0.05 * health["crp"]
)
health["deterioration_12h"] = (risk_score > np.percentile(risk_score, 75)).astype(int)

print(health.head())
print('positive rate:', round(health['deterioration_12h'].mean(), 3))


   heart_rate  resp_rate       spo2        crp  deterioration_12h
0  102.378642  25.510190  93.082322   0.204505                  1
1   79.168506  20.175046  97.499559  13.079440                  0
2   66.713524  15.893225  93.080655   6.499943                  0
3   57.082789  19.312968  93.095887   1.443583                  0
4  113.788002  22.378794  96.944844   6.224215                  1
positive rate: 0.25


## Domain Case Study C: Smart Cities (Traffic Congestion Proxy)

### Problem

Estimate congestion state from sensor counts and weather conditions.


In [4]:
k = 1000
traffic = pd.DataFrame({
    "vehicle_count": np.random.poisson(85, k),
    "avg_speed": np.random.normal(38, 9, k),
    "rain": np.random.binomial(1, 0.2, k),
    "incident_flag": np.random.binomial(1, 0.08, k),
})

traffic["congested"] = (
    (traffic["vehicle_count"] > 95) |
    (traffic["avg_speed"] < 28) |
    ((traffic["rain"] == 1) & (traffic["incident_flag"] == 1))
).astype(int)

print(traffic['congested'].value_counts(normalize=True).round(3))


congested
0    0.731
1    0.269
Name: proportion, dtype: float64


## Architecture Sketches (Text)

- **Healthcare**: EHR + notes -> feature/embedding service -> calibrated risk model -> clinician review UI -> audit logs.
- **Finance**: stream ingestion -> feature store -> rules + model ensemble -> case queue -> feedback loop.
- **Smart city**: edge sensors/cameras -> event stream -> forecasting/anomaly models -> control center -> fail-safe fallback plans.


## Connect to Theory

Each domain changes:

- what data is trustworthy,
- what errors are acceptable,
- which constraints dominate (regulation, safety, latency),
- how human oversight is implemented.


## Business Case Studies & Exceptions

### Case A: Clinical Decision Support

High recall may catch more at-risk patients, but excessive false alarms can overload clinicians. Thresholding and workflow integration matter as much as model architecture.

### Case B: Fraud Detection

A model that blocks too aggressively can hurt customer experience and revenue. Hybrid rule + model systems are often more robust than model-only pipelines.

### Case C: Traffic Optimization

Live optimization improves throughput but requires resilience to sensor outages and policy constraints from city authorities.

### Exceptions

- In strict compliance environments, interpretable models may be preferred even with lower peak accuracy.
- In early pilots, process redesign can generate more value than complex modeling.


## Interview Questions & Answers

1. **Q: How does domain context change model selection?**
   **A:** It changes risk tolerance, explainability needs, and latency/compliance constraints.
2. **Q: Why combine rules with ML in finance?**
   **A:** Rules enforce hard constraints while ML captures nuanced patterns.
3. **Q: What is a key healthcare AI constraint?**
   **A:** Patient safety and clinician trust with auditable decisions.
4. **Q: How do smart-city systems fail in production?**
   **A:** Sensor drift/outages, integration failures, and governance mismatches.
5. **Q: Why is AUC not enough in fraud systems?**
   **A:** Operational costs depend on thresholded decisions and false-positive burden.
6. **Q: What is a good domain-AI architecture interview answer?**
   **A:** Data -> model -> decision -> human oversight -> monitoring/feedback loop.
7. **Q: How do you adapt a generic model to a domain?**
   **A:** Add domain features, constraints, and evaluation tied to real KPIs.
8. **Q: What is the biggest non-model risk in domain AI?**
   **A:** Poor workflow integration with human operators.
9. **Q: When should you avoid deep models?**
   **A:** When data is limited, interpretability is critical, or latency budget is tiny.
10. **Q: How do you scale from toy to production?**
   **A:** Add data governance, robust evaluation, deployment controls, and incident playbooks.
